# Baseline Comparison — CodeBERT & RoBERTa vs CodeT5-small

**Goal:** Fine-tune CodeBERT and RoBERTa on the exact same train/test split used for CodeT5,
then compare accuracy, macro F1, and per-class F1 side by side.

**Models trained here:**
- `microsoft/codebert-base` — code-pretrained BERT, direct DL-to-DL comparison
- `roberta-base` — general language model, shows value of code-specific pretraining

**Your CodeT5 results (from prior evaluation):**
- Accuracy : 0.9984
- Macro F1 : 0.9985

**Hardware:** Apple M4 Pro — MPS backend

## 1. Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import json
import warnings
warnings.filterwarnings("ignore")

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
TRAIN_FILE   = "juliet_train_template_split.csv"
TEST_FILE    = "juliet_test_template_split.csv"
CODE_COL     = "code_clean"
LABEL_COL    = "cwe_index"           # numeric 0–N, same as CodeT5 training
CWE_COL      = "cwe"                 # string label, used for reporting only
NUM_CLASSES  = 56
MAX_LEN      = 256                   # same as CodeT5
BATCH_SIZE   = 16                    # safe for M4 Pro unified memory
EPOCHS       = 5
LR           = 2e-5
WARMUP_RATIO = 0.1
SEED         = 42

# Models to train — add or remove as needed
MODELS = {
    "CodeBERT": "microsoft/codebert-base",
    "RoBERTa":  "roberta-base",
}

# Your CodeT5 results to include in the final comparison table
CODET5_RESULTS = {
    "model":     "CodeT5-small (ours)",
    "accuracy":  0.9984,
    "macro_f1":  0.9985,
    "weighted_f1": 0.9984,
}

OUTPUT_DIR = "baseline_models"       # saved checkpoints go here
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ──────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
np.random.seed(SEED)

# Device — MPS for M4 Pro, fallback to CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print(f"Device : {DEVICE}")
print(f"Models : {list(MODELS.keys())}")
print(f"Classes: {NUM_CLASSES}")

## 2. Load & Validate Data

In [ ]:
train_df = pd.read_csv(TRAIN_FILE)[[CODE_COL, LABEL_COL, CWE_COL]].dropna()
test_df  = pd.read_csv(TEST_FILE)[[CODE_COL, LABEL_COL, CWE_COL]].dropna()

# Ensure labels are integers
train_df[LABEL_COL] = train_df[LABEL_COL].astype(int)
test_df[LABEL_COL]  = test_df[LABEL_COL].astype(int)

# Build index → CWE name mapping for readable reports
cwe_map = (
    pd.concat([train_df, test_df])[[LABEL_COL, CWE_COL]]
    .drop_duplicates()
    .sort_values(LABEL_COL)
)
idx_to_cwe = dict(zip(cwe_map[LABEL_COL], cwe_map[CWE_COL]))

# Verify label range matches NUM_CLASSES
all_labels = sorted(set(train_df[LABEL_COL]) | set(test_df[LABEL_COL]))
assert max(all_labels) < NUM_CLASSES, (
    f"Max label {max(all_labels)} >= NUM_CLASSES {NUM_CLASSES} — update NUM_CLASSES"
)

print(f"Train samples : {len(train_df):,}")
print(f"Test  samples : {len(test_df):,}")
print(f"Unique classes: {len(all_labels)}")
print(f"Label range   : {min(all_labels)} – {max(all_labels)}")
print()
print("Train class distribution (top 10):")
print(train_df[CWE_COL].value_counts().head(10).to_string())

## 3. Dataset Class

In [ ]:
class JulietDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df[CODE_COL].tolist()
        self.labels = df[LABEL_COL].tolist()
        self.tok    = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }

print("✅ Dataset class defined")

## 4. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for step, batch in enumerate(loader):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        # Gradient clipping — prevents exploding gradients on MPS
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        preds = outputs.logits.argmax(dim=-1)
        total_loss    += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total         += labels.size(0)

        if (step + 1) % 100 == 0:
            print(f"    step {step+1}/{len(loader)}  "
                  f"loss={total_loss/total:.4f}  "
                  f"acc={total_correct/total:.4f}")

    return total_loss / total, total_correct / total


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = outputs.logits.argmax(dim=-1).cpu()

            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


print("✅ Training & evaluation functions defined")

## 5. Run Baselines

Each model is:
1. Loaded from HuggingFace with a fresh classification head (56 classes)
2. Fine-tuned on `juliet_train_template_split.csv` for `EPOCHS` epochs
3. Evaluated on `juliet_test_template_split.csv`
4. Checkpoint saved to `baseline_models/<model_name>/`

**Estimated time on M4 Pro:**
- CodeBERT : ~60–90 min (125M params, MPS)
- RoBERTa  : ~60–90 min (125M params, MPS)

In [ ]:
import time

all_results = []   # collects final metrics per model
all_reports = {}   # collects full classification reports

for model_name, model_id in MODELS.items():
    print("="*68)
    print(f"  Training: {model_name}  ({model_id})")
    print("="*68)
    t0 = time.time()

    # ── Tokenizer & Datasets ──────────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    train_ds = JulietDataset(train_df, tokenizer, MAX_LEN)
    test_ds  = JulietDataset(test_df,  tokenizer, MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=0, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=False)

    # ── Model ─────────────────────────────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True,   # new classification head
    )
    model.to(DEVICE)

    # ── Optimizer & Scheduler ─────────────────────────────────────────────
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # ── Training loop ─────────────────────────────────────────────────────
    best_macro_f1   = 0.0
    best_model_path = os.path.join(OUTPUT_DIR, model_name)

    for epoch in range(1, EPOCHS + 1):
        print(f"\n  Epoch {epoch}/{EPOCHS}")
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)

        preds, labels_arr = evaluate(model, test_loader)
        val_acc      = accuracy_score(labels_arr, preds)
        val_macro_f1 = f1_score(labels_arr, preds, average="macro", zero_division=0)

        print(f"  → train loss={train_loss:.4f}  train acc={train_acc:.4f}  "
              f"val acc={val_acc:.4f}  val macro-F1={val_macro_f1:.4f}")

        # Save best checkpoint
        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            model.save_pretrained(best_model_path)
            tokenizer.save_pretrained(best_model_path)
            print(f"  ✅ New best — saved to {best_model_path}")

    # ── Final evaluation on best checkpoint ───────────────────────────────
    print(f"\n  Loading best checkpoint for final evaluation...")
    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_path)
    best_model.to(DEVICE)

    preds, labels_arr = evaluate(best_model, test_loader)

    accuracy    = accuracy_score(labels_arr, preds)
    macro_f1    = f1_score(labels_arr, preds, average="macro",    zero_division=0)
    weighted_f1 = f1_score(labels_arr, preds, average="weighted", zero_division=0)

    # Map numeric labels → CWE names for the report
    cwe_preds  = [idx_to_cwe.get(p, str(p)) for p in preds]
    cwe_labels = [idx_to_cwe.get(l, str(l)) for l in labels_arr]
    report     = classification_report(cwe_labels, cwe_preds, digits=4, zero_division=0)

    elapsed = (time.time() - t0) / 60

    print(f"\n  ── {model_name} Final Results ──")
    print(f"  Accuracy    : {accuracy:.4f}")
    print(f"  Macro F1    : {macro_f1:.4f}")
    print(f"  Weighted F1 : {weighted_f1:.4f}")
    print(f"  Time        : {elapsed:.1f} min")

    all_results.append({
        "model":       model_name,
        "accuracy":    round(accuracy, 4),
        "macro_f1":    round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
        "time_min":    round(elapsed, 1),
    })
    all_reports[model_name] = {
        "report": report,
        "preds":  preds.tolist(),
        "labels": labels_arr.tolist(),
    }

    # Free GPU/MPS memory before next model
    del model, best_model
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

print("\n✅ All baselines done")

## 6. Comparison Table

CodeT5-small results are added from your prior evaluation run.

In [ ]:
# Add your CodeT5 results
all_results.append({
    "model":       CODET5_RESULTS["model"],
    "accuracy":    CODET5_RESULTS["accuracy"],
    "macro_f1":    CODET5_RESULTS["macro_f1"],
    "weighted_f1": CODET5_RESULTS["weighted_f1"],
    "time_min":    None,
})

results_df = (
    pd.DataFrame(all_results)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)

# Highlight best score per metric
print("="*68)
print("  BASELINE COMPARISON — Juliet Multiclass CWE Classification")
print(f"  Dataset : {len(train_df):,} train / {len(test_df):,} test samples")
print(f"  Classes : {NUM_CLASSES}")
print("="*68)
print()
print(results_df.to_string(index=False))
print()

# Find improvements of CodeT5 over each baseline
codet5_row = results_df[results_df["model"].str.contains("CodeT5")].iloc[0]
print("  Improvement of CodeT5 over baselines:")
for _, row in results_df.iterrows():
    if "CodeT5" not in row["model"]:
        delta_acc = (codet5_row["accuracy"] - row["accuracy"]) * 100
        delta_f1  = (codet5_row["macro_f1"] - row["macro_f1"]) * 100
        print(f"    vs {row['model']:<12}  "
              f"accuracy +{delta_acc:.2f}pp  macro-F1 +{delta_f1:.2f}pp")

## 7. Per-Class F1 Comparison

Shows exactly which CWEs each model struggles with — important for your confused-pairs analysis.

In [ ]:
from sklearn.metrics import classification_report as cr

per_class_rows = []

for model_name, data in all_reports.items():
    report_dict = cr(
        [idx_to_cwe.get(l, str(l)) for l in data["labels"]],
        [idx_to_cwe.get(p, str(p)) for p in data["preds"]],
        output_dict=True,
        zero_division=0,
    )
    for cwe, metrics in report_dict.items():
        if cwe in ("accuracy", "macro avg", "weighted avg"):
            continue
        if isinstance(metrics, dict):
            per_class_rows.append({
                "model":     model_name,
                "cwe":       cwe,
                "precision": round(metrics["precision"], 4),
                "recall":    round(metrics["recall"],    4),
                "f1":        round(metrics["f1-score"],  4),
                "support":   int(metrics["support"]),
            })

per_class_df = pd.DataFrame(per_class_rows)

# Pivot: one row per CWE, columns = model F1 scores
pivot = per_class_df.pivot_table(
    index="cwe", columns="model", values="f1"
).round(4)

# Add support from either model (same test set)
support = per_class_df.groupby("cwe")["support"].first()
pivot["support"] = support
pivot = pivot.sort_values(list(MODELS.keys())[0])  # sort by worst CodeBERT F1

print("Per-class F1 comparison (sorted by CodeBERT F1, ascending):")
print(pivot.to_string())

## 8. Confusion Analysis on the Same 16 Pairs

For each of your 16 significantly confused pairs, compare the error rates across all models.
This directly shows whether CodeT5 is better at the *specific* failure modes you identified.

In [ ]:
# Your 16 significant confused pairs from the confusion analysis
CONFUSED_PAIRS = [
    ("CWE188", "CWE190"), ("CWE176", "CWE190"), ("CWE121", "CWE123"),
    ("CWE122", "CWE123"), ("CWE114", "CWE127"), ("CWE121", "CWE190"),
    ("CWE122", "CWE190"), ("CWE123", "CWE15"),  ("CWE124", "CWE190"),
    ("CWE126", "CWE15"),  ("CWE134", "CWE15"),  ("CWE134", "CWE190"),
    ("CWE15",  "CWE190"), ("CWE176", "CWE15"),  ("CWE188", "CWE15"),
    ("CWE244", "CWE15"),
]

pair_comparison = []

for model_name, data in all_reports.items():
    preds_cwe  = [idx_to_cwe.get(p, str(p)) for p in data["preds"]]
    labels_cwe = [idx_to_cwe.get(l, str(l)) for l in data["labels"]]

    preds_s  = pd.Series(preds_cwe)
    labels_s = pd.Series(labels_cwe)

    for true_cwe, pred_cwe in CONFUSED_PAIRS:
        true_mask  = labels_s == true_cwe
        total_true = true_mask.sum()
        confused   = ((labels_s == true_cwe) & (preds_s == pred_cwe)).sum()
        rate       = confused / total_true if total_true > 0 else 0.0

        pair_comparison.append({
            "model":        model_name,
            "True_CWE":     true_cwe,
            "Predicted_CWE":pred_cwe,
            "confused_n":   int(confused),
            "total_n":      int(total_true),
            "error_rate":   round(float(rate), 6),
        })

pair_df = pd.DataFrame(pair_comparison)

# Pivot: one row per pair, columns = error rate per model
pair_pivot = pair_df.pivot_table(
    index=["True_CWE", "Predicted_CWE"],
    columns="model",
    values="error_rate",
).round(6)

# Add CodeT5 error rates from your confusion analysis
codet5_rates = {
    ("CWE188","CWE190"): 0.013514, ("CWE176","CWE190"): 0.010811,
    ("CWE121","CWE123"): 0.002158, ("CWE122","CWE123"): 0.001660,
    ("CWE114","CWE127"): 0.009009, ("CWE121","CWE190"): 0.000719,
    ("CWE122","CWE190"): 0.000553, ("CWE123","CWE15"):  0.001252,
    ("CWE124","CWE190"): 0.002105, ("CWE126","CWE15"):  0.001252,
    ("CWE134","CWE15"):  0.000644, ("CWE134","CWE190"): 0.000644,
    ("CWE15", "CWE190"): 0.001152, ("CWE176","CWE15"):  0.002703,
    ("CWE188","CWE15"):  0.002703, ("CWE244","CWE15"):  0.004386,
}
pair_pivot["CodeT5-small (ours)"] = [
    codet5_rates.get(idx, 0.0) for idx in pair_pivot.index
]

print("Error rates on 16 significant confused pairs (lower is better):")
print()
print(pair_pivot.to_string())

## 9. Export All Results

In [ ]:
# 1. Summary comparison table
results_df.to_csv("baseline_comparison_summary.csv", index=False)

# 2. Per-class F1 pivot
pivot.to_csv("baseline_per_class_f1.csv")

# 3. Confused pairs error rate comparison
pair_pivot.to_csv("baseline_confused_pairs_error_rates.csv")

# 4. Full classification reports as text
with open("baseline_classification_reports.txt", "w") as f:
    for model_name, data in all_reports.items():
        f.write(f"{'='*68}\n")
        f.write(f"{model_name}\n")
        f.write(f"{'='*68}\n")
        f.write(data["report"] + "\n\n")

print("✅ Exported:")
print("   baseline_comparison_summary.csv       — accuracy / macro-F1 / weighted-F1")
print("   baseline_per_class_f1.csv             — per-CWE F1 for each model")
print("   baseline_confused_pairs_error_rates.csv — error rates on 16 confused pairs")
print("   baseline_classification_reports.txt   — full classification reports")